# 특이값 분해 (SVD)

> 선형대수 16강 · 특이값 분해

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [특이값 분해 (SVD)](https://mioon1402.github.io/timeseriesdata/linalg/L16-svd.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.**

In [ ]:

print('준비 완료')

## 0. 기초 다지기 — 고유분해의 한계

## 1. 단위원 → 타원

## 2. A = UΣVᵀ — 회전·늘림·회전

## 3. 어떻게 찾는가 — AᵀA 의 힘

## 4. 특이값이 알려주는 것

## 5. SVD 와 네 부분공간

## 6. numpy 로 확인하기

**16-1. SVD 계산과 복원**

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

A = np.array([[3., 0.],
              [4., 5.]])

U, σ, Vt = np.linalg.svd(A)      # 주의: 세 번째는 V 가 아니라 Vᵀ 다

print("U =")
print(U)
print("\nσ =", σ, "  ← 내림차순, 전부 0 이상")
print("\nVᵀ =")
print(Vt)
print()
print("복원 U @ diag(σ) @ Vᵀ =")
print(U @ np.diag(σ) @ Vt)
print("A 와 같은가:", np.allclose(U @ np.diag(σ) @ Vt, A))

**16-2. Avᵢ = σᵢuᵢ 확인**

In [ ]:
for i in range(2):
    v = Vt[i]                    # i번째 오른쪽 특이벡터 (행!)
    u = U[:, i]                  # i번째 왼쪽 특이벡터 (열!)
    print(f"i={i}:  A@v = {np.round(A @ v, 4)}   σ·u = {np.round(σ[i] * u, 4)}   "
          f"같은가: {np.allclose(A @ v, σ[i] * u)}")

print()
print("UᵀU =", np.round(U.T @ U, 10).tolist(), " ← 직교")
print("VᵀV =", np.round(Vt @ Vt.T, 10).tolist(), " ← 직교")

**16-3. 원이 정말 타원이 되는가**

In [ ]:
각도 = np.linspace(0, 2*np.pi, 8, endpoint=False)
원 = np.column_stack([np.cos(각도), np.sin(각도)])     # 단위원 위 8개 점

타원 = 원 @ A.T                                        # 각 점에 A 를 적용

print(f"{'입력(길이 1)':>22}  {'출력':>22}  {'길이':>8}")
for i in range(len(원)):
    print(f"{str(np.round(원[i], 3)):>22}  {str(np.round(타원[i], 3)):>22}  {np.linalg.norm(타원[i]):8.4f}")

print()
print(f"출력 길이의 최대 = {np.linalg.norm(타원, axis=1).max():.4f}  (σ₁ = {σ[0]:.4f} 에 근접)")
print(f"           최소 = {np.linalg.norm(타원, axis=1).min():.4f}  (σ₂ = {σ[1]:.4f} 이상)")
print("→ 점을 촘촘히 찍을수록 정확히 σ₁, σ₂ 에 도달한다")

**16-4. σ² 는 AᵀA 의 고윳값**

In [ ]:
AtA = A.T @ A
print("AᵀA =")
print(AtA, "  대칭:", np.allclose(AtA, AtA.T))
print()
print("AᵀA 의 고윳값 =", np.sort(np.linalg.eigvalsh(AtA))[::-1])
print("σ²          =", σ ** 2)
print("같은가:", np.allclose(np.sort(np.linalg.eigvalsh(AtA))[::-1], σ ** 2))
print()
print("AAᵀ 의 고윳값 =", np.sort(np.linalg.eigvalsh(A @ A.T))[::-1], " ← 같다")
print("→ AᵀA 와 AAᵀ 는 크기가 달라도 0 아닌 고윳값이 동일하다")

**16-5. 직사각·특이 행렬도 전부 된다**

In [ ]:
행렬들 = {
    "정사각 가역"   : np.array([[3., 0.], [4., 5.]]),
    "키가 큰 4×2"   : np.array([[1., 0.], [0., 1.], [1., 1.], [2., -1.]]),
    "납작한 2×4"    : np.array([[1., 2., 3., 4.], [5., 6., 7., 8.]]),
    "랭크 1 (특이)" : np.array([[1., 2., 3.], [2., 4., 6.]]),
    "영행렬"        : np.zeros((2, 3)),
}

for 이름, Mx in 행렬들.items():
    U2, s2, Vt2 = np.linalg.svd(Mx)
    복원 = U2[:, :len(s2)] @ np.diag(s2) @ Vt2[:len(s2)]
    print(f"{이름:16s} {str(Mx.shape):8s} σ = {np.round(s2, 4)}")
    print(f"{'':16s} 랭크 {np.linalg.matrix_rank(Mx)}   복원 정확: {np.allclose(복원, Mx)}")
print()
print("→ 어떤 모양이든, 특이해도, 영행렬이어도 SVD 는 존재한다")

**16-6. 특이값이 알려주는 것들**

In [ ]:
B = np.array([[3., 0.], [4., 5.]])
U3, s3, Vt3 = np.linalg.svd(B)

print("σ =", s3)
print()
print("랭크         =", int((s3 > 1e-10).sum()), " (numpy:", np.linalg.matrix_rank(B), ")")
print("‖B‖₂ (최대확대) =", s3[0], " (numpy:", round(np.linalg.norm(B, 2), 6), ")")
print("조건수        =", s3[0] / s3[-1], " (numpy:", round(np.linalg.cond(B), 6), ")")
print("|det|        =", np.prod(s3), " (numpy:", round(abs(np.linalg.det(B)), 6), ")")

**16-7. 네 부분공간의 기저**

In [ ]:
C = np.array([[1., 2., 3.],
              [2., 4., 6.],
              [1., 1., 1.]])
U4, s4, Vt4 = np.linalg.svd(C)
r = int((s4 > 1e-10).sum())
m, n = C.shape

print("σ =", np.round(s4, 6), "  랭크 r =", r)
print()
print(f"열공간   기저 = U 의 앞 {r}열")
print(np.round(U4[:, :r], 4))
print(f"\n좌영공간 기저 = U 의 나머지 {m-r}열")
print(np.round(U4[:, r:], 4))
print(f"\n행공간   기저 = Vᵀ 의 앞 {r}행")
print(np.round(Vt4[:r], 4))
print(f"\n영공간   기저 = Vᵀ 의 나머지 {n-r}행")
print(np.round(Vt4[r:], 4))
print()
print("확인: C @ (영공간 기저) =", np.round(C @ Vt4[r:].T, 10).ravel())

**16-8. 연습문제**

In [ ]:
# 문제 1. 대각행렬 [[3,0],[0,2]] 의 SVD 는 어떤 모양일까요?
#         (힌트: 이미 '늘이기' 만 하고 있다)

# 문제 2. 회전행렬의 특이값은? 왜 그럴까요?

# 문제 3. 대칭 양의 정부호 행렬의 SVD 와 고유분해는 어떤 관계일까요?
#         [[4,1],[1,3]] 으로 확인해보세요.

# 아래에 직접 써보세요

**모범 답안**

In [ ]:
# 문제 1 — U 와 V 는 (부호를 빼면) 단위행렬, σ 는 대각 원소
D = np.array([[3., 0.], [0., 2.]])
U1, s1, V1 = np.linalg.svd(D)
print("문제 1: σ =", s1, "\nU =\n", U1, "\nVᵀ =\n", V1)
print("→ 이미 축 방향으로만 늘이고 있어 회전이 필요 없다")

# 문제 2 — 회전은 길이를 보존하므로 모든 σ = 1
θ = np.deg2rad(37)
R = np.array([[np.cos(θ), -np.sin(θ)], [np.sin(θ), np.cos(θ)]])
print("\n문제 2: σ =", np.round(np.linalg.svd(R)[1], 10))
print("→ 원이 원으로 간다. 늘어나는 방향이 없으므로 전부 1 (11강)")

# 문제 3 — PD 대칭이면 σ = λ, U = V = Q
S3 = np.array([[4., 1.], [1., 3.]])
λ3, Q3 = np.linalg.eigh(S3)
U3, s3, Vt3 = np.linalg.svd(S3)
print("\n문제 3: 고윳값 =", np.sort(λ3)[::-1], "   σ =", s3)
print("        |U| 와 |V| 가 같은가:", np.allclose(np.abs(U3), np.abs(Vt3.T)))
print("→ 양의 정부호 대칭행렬에서는 SVD 와 고유분해가 사실상 같다.")
print("   고윳값이 음수면 σ = |λ| 가 되고 부호가 U 로 흡수된다.")

---

전체 강의 목록 → [눈으로 보는 수학·통계](https://mioon1402.github.io/timeseriesdata/)